# FoundationPose Evaluation — Colab GPU

**Ejecutar DESPUES de `00_colab_setup.ipynb`**

Este notebook:
1. Instala FoundationPose desde el repo oficial de NVIDIA
2. Descarga pesos pre-entrenados (scorer + refiner)
3. Ejecuta inferencia en YCB-V y T-LESS (una pose por objeto)
4. Calcula metricas BOP (ADD, ADD-S, MSSD, MSPD)
5. Guarda resultados en Google Drive

**Requiere:** GPU T4 o superior (Runtime > Change runtime type > T4 GPU)

**Referencia:** Wen et al., *FoundationPose: Unified 6D Pose Estimation and Tracking of Novel Objects*, CVPR 2024

In [ ]:
import torch
import os
import sys

assert torch.cuda.is_available(), "GPU requerida! Ve a Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

REPO_DIR = "/content/repo_tfm"
assert os.path.exists(REPO_DIR), "Ejecuta 00_colab_setup.ipynb primero"
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

## 1. Instalar FoundationPose

FoundationPose requiere:
- `nvdiffrast` (NVIDIA differentiable rasterizer) — necesita CUDA
- `mycuda` extensiones C++/CUDA del repo
- `trimesh`, `opencv`, `scipy`, `pyrender`

> **Nota:** La compilacion de extensiones CUDA puede tardar 2-3 min.

In [ ]:
FP_DIR = "/content/FoundationPose"

if not os.path.exists(FP_DIR):
    print("Clonando FoundationPose...")
    !git clone --depth 1 https://github.com/NVlabs/FoundationPose.git {FP_DIR}
    print("OK")
else:
    print(f"FoundationPose ya existe en {FP_DIR}")

# Dependencias de FoundationPose (sin open3d ni pyrender — no se necesitan)
print("\nInstalando dependencias...")
!pip install -q trimesh opencv-python-headless scipy scikit-learn
!pip install -q ruamel.yaml

# nvdiffrast (NVIDIA differentiable rasterizer — requiere CUDA)
print("\nInstalando nvdiffrast...")
!pip install -q git+https://github.com/NVlabs/nvdiffrast

# Compilar extensiones CUDA de FoundationPose
print("\nCompilando extensiones CUDA de FoundationPose...")
!cd {FP_DIR} && pip install -q -e mycuda 2>&1 | tail -5
!cd {FP_DIR} && pip install -q -e bundlesdf/mycuda 2>&1 | tail -5

sys.path.insert(0, FP_DIR)

# Verificar que numpy no fue downgradeado
import numpy as np
print(f"\nnumpy: {np.__version__}")

try:
    import nvdiffrast.torch as dr
    print("[OK] nvdiffrast")
except ImportError as e:
    print(f"[ERROR] nvdiffrast: {e}")

try:
    import trimesh
    print("[OK] trimesh")
except ImportError as e:
    print(f"[ERROR] trimesh: {e}")

print(f"\nFoundationPose en: {FP_DIR}")


## 2. Descargar pesos pre-entrenados

FoundationPose usa dos modelos:
- **ScorePredictor** — evalua calidad de hipotesis de pose
- **PoseRefinePredictor** — refina iterativamente la pose

Los pesos se descargan del enlace oficial y se colocan en los directorios
que el codigo espera internamente (`weights/` dentro del repo).

> Consulta: https://github.com/NVlabs/FoundationPose#pre-trained-weights

In [ ]:
# Google Drive para persistir pesos entre sesiones
USE_GDRIVE = os.path.exists('/content/drive/MyDrive')
if not USE_GDRIVE:
    print("Montando Google Drive para persistir pesos...")
    from google.colab import drive
    drive.mount('/content/drive')
    USE_GDRIVE = os.path.exists('/content/drive/MyDrive')

# Directorio persistente para pesos
GDRIVE_WEIGHTS = "/content/drive/MyDrive/TFM/weights/foundationpose"
os.makedirs(GDRIVE_WEIGHTS, exist_ok=True)

# FoundationPose espera los pesos en estas rutas relativas al repo:
#   - weights/2024-01-11-20-02-45/  (scorer)
#   - weights/2023-10-28-18-33-37/  (refiner, via bundlesdf)
# Ver: https://github.com/NVlabs/FoundationPose#pre-trained-weights

FP_WEIGHTS_DIR = f"{FP_DIR}/weights"
os.makedirs(FP_WEIGHTS_DIR, exist_ok=True)

# Pesos del scorer
SCORER_DIR = f"{FP_WEIGHTS_DIR}/2024-01-11-20-02-45"
# Pesos del refiner (bundlesdf)
REFINER_DIR = f"{FP_DIR}/bundlesdf/ckpts/2023-10-28-18-33-37"

print("=" * 60)
print("Estado de pesos pre-entrenados de FoundationPose")
print("=" * 60)

# Verificar scorer
scorer_ok = os.path.exists(SCORER_DIR) and len(os.listdir(SCORER_DIR)) > 0
if scorer_ok:
    print(f"[OK] Scorer: {SCORER_DIR}")
else:
    print(f"[!] Scorer NO encontrado en: {SCORER_DIR}")

# Verificar refiner
refiner_ok = os.path.exists(REFINER_DIR) and len(os.listdir(REFINER_DIR)) > 0
if refiner_ok:
    print(f"[OK] Refiner: {REFINER_DIR}")
else:
    print(f"[!] Refiner NO encontrado en: {REFINER_DIR}")

if not scorer_ok or not refiner_ok:
    print("\n" + "=" * 60)
    print("INSTRUCCIONES PARA DESCARGAR PESOS:")
    print("=" * 60)
    print("")
    print("1. Ve a: https://github.com/NVlabs/FoundationPose#pre-trained-weights")
    print("2. Descarga los dos archivos .zip de pesos")
    print("3. Subelos a Google Drive en:")
    print(f"   {GDRIVE_WEIGHTS}/")
    print("4. O ejecuta la celda siguiente para copiarlos desde Drive")
    print("")
    print("Alternativamente, usa gdown si tienes el enlace directo:")
    print('   !pip install -q gdown')
    print('   !gdown --id <FILE_ID> -O weights.zip')
else:
    print("\nTodos los pesos disponibles. Listo para inferencia.")

In [ ]:
# --- OPCIONAL: Copiar pesos desde Google Drive al repo ---
# Ejecutar solo si subiste los pesos a Google Drive manualmente

import shutil

def setup_weights_from_gdrive():
    """Copia y extrae pesos de Google Drive a las rutas esperadas por FoundationPose."""
    import zipfile
    
    gdrive_files = os.listdir(GDRIVE_WEIGHTS) if os.path.exists(GDRIVE_WEIGHTS) else []
    print(f"Archivos en Google Drive: {gdrive_files}")
    
    for f in gdrive_files:
        src = os.path.join(GDRIVE_WEIGHTS, f)
        if f.endswith('.zip'):
            print(f"Extrayendo {f}...")
            with zipfile.ZipFile(src, 'r') as zf:
                # Extraer en el directorio de pesos del repo
                zf.extractall(FP_WEIGHTS_DIR)
            print(f"  Extraido en {FP_WEIGHTS_DIR}")
        elif os.path.isdir(src):
            dst = os.path.join(FP_WEIGHTS_DIR, f)
            if not os.path.exists(dst):
                shutil.copytree(src, dst)
                print(f"  Copiado {f} -> {dst}")
    
    # Verificar de nuevo
    print(f"\nContenido de {FP_WEIGHTS_DIR}:")
    !ls -la {FP_WEIGHTS_DIR}

# Descomentar para ejecutar:
# setup_weights_from_gdrive()

## 3. Verificar datasets BOP

In [ ]:
import numpy as np
import json
import time
from pathlib import Path

DATA_DIR = f"{REPO_DIR}/data/datasets"
print(f"Verificando datasets en: {DATA_DIR}\n")

for ds in ['ycbv', 'tless']:
    p = Path(DATA_DIR) / ds
    if p.exists():
        # Modelos 3D
        models_dir = p / 'models'
        models = sorted(models_dir.glob('obj_*.ply')) if models_dir.exists() else []
        print(f"{ds.upper()}: {len(models)} modelos 3D")
        
        # Escenas de test
        for split in ['test', 'test_primesense']:
            split_dir = p / split
            if split_dir.exists():
                scenes = sorted([d for d in split_dir.iterdir() if d.is_dir()])
                print(f"  {split}: {len(scenes)} escenas")
                if scenes:
                    # Contar imagenes en primera escena
                    rgb_dir = scenes[0] / 'rgb'
                    if rgb_dir.exists():
                        n_imgs = len(list(rgb_dir.glob('*.png')))
                        print(f"    Escena {scenes[0].name}: {n_imgs} imagenes")
    else:
        print(f"{ds.upper()}: NO encontrado (ejecuta 00_colab_setup.ipynb primero)")

## 4. Inicializar FoundationPose

Flujo de la API oficial (Wen et al., CVPR 2024):

```python
scorer = ScorePredictor()           # Evalua hipotesis de pose
refiner = PoseRefinePredictor()     # Refina iterativamente
glctx = dr.RasterizeCudaContext()   # Rasterizador GPU

est = FoundationPose(
    model_pts=mesh.vertices,         # Vertices del modelo 3D
    model_normals=mesh.vertex_normals,
    mesh=mesh,                       # trimesh.Trimesh
    scorer=scorer, refiner=refiner, glctx=glctx
)

pose_4x4 = est.register(K=K, rgb=rgb, depth=depth, ob_mask=mask, iteration=5)
```

In [ ]:
import trimesh
import nvdiffrast.torch as dr

# Importar modulos de FoundationPose
sys.path.insert(0, FP_DIR)
from estimater import FoundationPose, ScorePredictor, PoseRefinePredictor

# --- Inicializar los modelos (una sola vez) ---
print("Cargando ScorePredictor...")
scorer = ScorePredictor()
print("[OK] ScorePredictor cargado")

print("Cargando PoseRefinePredictor...")
refiner = PoseRefinePredictor()
print("[OK] PoseRefinePredictor cargado")

print("Creando contexto de rasterizacion CUDA...")
glctx = dr.RasterizeCudaContext()
print("[OK] nvdiffrast glctx creado")

print("\nFoundationPose listo para inferencia.")

In [ ]:
# --- Cargar meshes de los objetos ---
# BOP format: data/datasets/{ycbv,tless}/models/obj_XXXXXX.ply

def load_bop_meshes(dataset_dir):
    """Carga todos los meshes de un dataset BOP.
    
    Returns:
        dict: {obj_id: trimesh.Trimesh}
    """
    models_dir = Path(dataset_dir) / 'models'
    meshes = {}
    
    for ply_file in sorted(models_dir.glob('obj_*.ply')):
        # obj_000001.ply -> obj_id = 1
        obj_id = int(ply_file.stem.split('_')[1])
        mesh = trimesh.load(str(ply_file), process=False)
        
        # BOP meshes estan en mm, convertir a metros
        mesh.vertices *= 1e-3
        
        meshes[obj_id] = mesh
    
    return meshes


def create_estimator(mesh, scorer, refiner, glctx):
    """Crea un estimador FoundationPose para un objeto dado."""
    est = FoundationPose(
        model_pts=mesh.vertices.copy(),
        model_normals=mesh.vertex_normals.copy(),
        mesh=mesh,
        scorer=scorer,
        refiner=refiner,
        glctx=glctx,
        debug=0,
    )
    return est


# Cargar meshes YCB-V
ycbv_meshes = load_bop_meshes(f"{DATA_DIR}/ycbv")
print(f"YCB-V: {len(ycbv_meshes)} meshes cargados (obj_ids: {sorted(ycbv_meshes.keys())})")

# Cargar meshes T-LESS (si disponibles)
tless_path = Path(DATA_DIR) / 'tless'
if (tless_path / 'models').exists():
    tless_meshes = load_bop_meshes(f"{DATA_DIR}/tless")
    print(f"T-LESS: {len(tless_meshes)} meshes cargados")
else:
    tless_meshes = {}
    print("T-LESS: modelos no disponibles")

## 5. Ejecutar FoundationPose en YCB-Video

Para cada imagen de test:
1. Cargar RGB, depth, K (intrinsicos)
2. Para cada objeto en la escena (segun GT):
   - Cargar su mesh 3D
   - Cargar su mascara (GT o segmentacion)
   - `est.register()` → pose 4x4
3. Guardar predicciones para evaluacion BOP

In [ ]:
from src.utils.dataset_loader import BOPDataset
from tqdm.notebook import tqdm
import cv2

# Cargar dataset YCB-V
print(f"Cargando YCB-V desde {DATA_DIR}/ycbv")
ycbv = BOPDataset(f"{DATA_DIR}/ycbv", split="test")
ycbv_scenes = ycbv.get_scene_ids()
print(f"YCB-V: {len(ycbv_scenes)} escenas de test")

# --- Configuracion de evaluacion ---
MAX_SCENES = 5          # None para evaluar todas
MAX_IMAGES_PER_SCENE = 50
REGISTER_ITERATIONS = 5  # Iteraciones de refinamiento (5 = default paper)

eval_scenes = ycbv_scenes[:MAX_SCENES] if MAX_SCENES else ycbv_scenes
print(f"\nEvaluando {len(eval_scenes)} escenas (MAX_SCENES={MAX_SCENES})")
print(f"Iteraciones de registro: {REGISTER_ITERATIONS}")

In [ ]:
# --- Inferencia con CHECKPOINTS ---
# Guarda progreso cada N escenas. Si Colab se desconecta,
# re-ejecuta esta celda y retoma automaticamente.

CHECKPOINT_EVERY = 2  # Guardar cada N escenas
CHECKPOINT_DIR = "/content/drive/MyDrive/TFM/checkpoints" if USE_GDRIVE else "/content/checkpoints"
CHECKPOINT_FILE = f"{CHECKPOINT_DIR}/fp_ycbv_checkpoint.json"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Intentar retomar desde checkpoint
results_ycbv = []
completed_scenes = set()
timing_total = 0
n_objects_evaluated = 0
n_images_evaluated = 0
errors = []

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE) as f:
        ckpt = json.load(f)
    results_ycbv = ckpt["results"]
    completed_scenes = set(ckpt["completed_scenes"])
    timing_total = ckpt["timing_total"]
    n_objects_evaluated = ckpt["n_objects_evaluated"]
    n_images_evaluated = ckpt["n_images_evaluated"]
    print(f"CHECKPOINT ENCONTRADO: {len(completed_scenes)} escenas ya completadas")
    print(f"  Retomando desde escena siguiente...\n")
else:
    print("Sin checkpoint previo. Comenzando desde cero.\n")

estimator_cache = {}

for scene_idx, scene_id in enumerate(tqdm(eval_scenes, desc="Escenas YCB-V")):
    # Saltar escenas ya completadas
    if scene_id in completed_scenes:
        continue

    n_images = ycbv.get_num_images(scene_id)
    scene_gt = ycbv.load_scene_gt(scene_id)
    scene_camera = ycbv.load_scene_camera(scene_id)

    image_ids = ycbv.get_image_ids(scene_id)
    for img_id in image_ids[:MAX_IMAGES_PER_SCENE]:
        try:
            sample = ycbv.load_sample(scene_id, img_id)
            rgb = sample["rgb"]
            depth = sample["depth"]
            cam_K = sample["cam_K"]

            img_key = str(img_id)
            gt_list = scene_gt.get(img_key, scene_gt.get(img_id, []))

            for gt_idx, gt in enumerate(gt_list):
                obj_id = gt["obj_id"]
                if obj_id not in ycbv_meshes:
                    continue

                mask = sample.get("mask", None)
                if mask is None:
                    mask_path = Path(DATA_DIR) / "ycbv" / "test" / scene_id / "mask_visib" / f"{img_id:06d}_{gt_idx:06d}.png"
                    if mask_path.exists():
                        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
                        mask = (mask > 0).astype(np.uint8)
                    else:
                        continue
                elif mask.ndim == 2:
                    mask = (mask > 0).astype(np.uint8)

                if obj_id not in estimator_cache:
                    estimator_cache[obj_id] = create_estimator(
                        ycbv_meshes[obj_id], scorer, refiner, glctx
                    )
                est = estimator_cache[obj_id]

                t0 = time.time()
                pose_4x4 = est.register(
                    K=cam_K, rgb=rgb, depth=depth,
                    ob_mask=mask, iteration=REGISTER_ITERATIONS,
                )
                elapsed = time.time() - t0
                timing_total += elapsed
                n_objects_evaluated += 1

                R_pred = pose_4x4[:3, :3]
                t_pred = pose_4x4[:3, 3]

                results_ycbv.append({
                    "scene_id": int(scene_id),
                    "img_id": int(img_id),
                    "obj_id": int(obj_id),
                    "R_pred": R_pred.tolist(),
                    "t_pred": t_pred.tolist(),
                    "time_s": elapsed,
                })

            n_images_evaluated += 1

        except Exception as e:
            err_msg = f"Scene {scene_id} img {img_id}: {str(e)[:150]}"
            errors.append(err_msg)
            if len(errors) <= 5:
                print(f"\n[Error] {err_msg}")
            continue

    completed_scenes.add(scene_id)

    # --- CHECKPOINT: guardar progreso ---
    if (scene_idx + 1) % CHECKPOINT_EVERY == 0:
        ckpt_data = {
            "results": results_ycbv,
            "completed_scenes": list(completed_scenes),
            "timing_total": timing_total,
            "n_objects_evaluated": n_objects_evaluated,
            "n_images_evaluated": n_images_evaluated,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        with open(CHECKPOINT_FILE, "w") as f:
            json.dump(ckpt_data, f)
        print(f"  [CKPT] Guardado: {len(completed_scenes)} escenas, {n_objects_evaluated} objetos")

# Checkpoint final
if results_ycbv:
    ckpt_data = {
        "results": results_ycbv,
        "completed_scenes": list(completed_scenes),
        "timing_total": timing_total,
        "n_objects_evaluated": n_objects_evaluated,
        "n_images_evaluated": n_images_evaluated,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "status": "COMPLETED",
    }
    with open(CHECKPOINT_FILE, "w") as f:
        json.dump(ckpt_data, f)

# --- Resumen ---
avg_time = timing_total / max(n_objects_evaluated, 1)
print(f"\n{'=' * 60}")
print(f"  FoundationPose YCB-Video — Resumen")
print(f"{'=' * 60}")
print(f"  Imagenes evaluadas:    {n_images_evaluated}")
print(f"  Objetos evaluados:     {n_objects_evaluated}")
print(f"  Predicciones totales:  {len(results_ycbv)}")
print(f"  Tiempo promedio/obj:   {avg_time*1000:.1f} ms")
if avg_time > 0:
    print(f"  Throughput:            {1/avg_time:.1f} objetos/s")
if errors:
    print(f"  Errores:               {len(errors)}")
print(f"  Checkpoint:            {CHECKPOINT_FILE}")
print(f"{'=' * 60}")


## 6. Calcular metricas BOP

In [ ]:
from src.utils.metrics import add_metric, add_s_metric, compute_recall, compute_auc

if results_ycbv:
    # Calcular ADD y ADD-S comparando con GT
    add_errors = []
    adds_errors = []
    
    for pred in tqdm(results_ycbv, desc="Calculando metricas"):
        scene_id = pred['scene_id']
        img_id = pred['img_id']
        obj_id = pred['obj_id']
        
        # Obtener GT pose
        scene_gt = ycbv.load_scene_gt(scene_id)
        img_key = str(img_id)
        gt_list = scene_gt.get(img_key, scene_gt.get(img_id, []))
        
        gt_pose = None
        for gt in gt_list:
            if gt['obj_id'] == obj_id:
                R_gt = np.array(gt['cam_R_m2c']).reshape(3, 3)
                t_gt = np.array(gt['cam_t_m2c']).reshape(3) * 1e-3  # mm -> m
                gt_pose = (R_gt, t_gt)
                break
        
        if gt_pose is None:
            continue
        
        R_pred = np.array(pred['R_pred'])
        t_pred = np.array(pred['t_pred'])
        
        # Puntos del modelo
        if obj_id in ycbv_meshes:
            pts = ycbv_meshes[obj_id].vertices  # Ya en metros
            
            # ADD
            add_err = add_metric(R_pred, t_pred, gt_pose[0], gt_pose[1], pts)
            add_errors.append(add_err)
            
            # ADD-S (simetrico)
            adds_err = add_s_metric(R_pred, t_pred, gt_pose[0], gt_pose[1], pts)
            adds_errors.append(adds_err)
    
    # Calcular recall y AUC
    add_errors = np.array(add_errors)
    adds_errors = np.array(adds_errors)
    
    thresholds_mm = [5, 10, 20, 50]  # en mm
    
    print(f"\n{'=' * 60}")
    print(f"  FoundationPose — YCB-Video Metrics")
    print(f"{'=' * 60}")
    print(f"  Objetos evaluados: {len(add_errors)}")
    print(f"")
    print(f"  ADD  (mean):     {add_errors.mean()*1000:.2f} mm")
    print(f"  ADD  (median):   {np.median(add_errors)*1000:.2f} mm")
    print(f"  ADD-S (mean):    {adds_errors.mean()*1000:.2f} mm")
    print(f"  ADD-S (median):  {np.median(adds_errors)*1000:.2f} mm")
    print(f"")
    
    for th in thresholds_mm:
        recall_add = (add_errors * 1000 < th).mean() * 100
        recall_adds = (adds_errors * 1000 < th).mean() * 100
        print(f"  Recall@{th}mm  ADD: {recall_add:.1f}%  ADD-S: {recall_adds:.1f}%")
    
    # AUC (area bajo la curva recall vs threshold)
    auc_add = compute_auc(add_errors * 1000, max_threshold=100)  # hasta 100mm
    auc_adds = compute_auc(adds_errors * 1000, max_threshold=100)
    print(f"")
    print(f"  AUC ADD:   {auc_add:.4f}")
    print(f"  AUC ADD-S: {auc_adds:.4f}")
    print(f"{'=' * 60}")
    
    # Guardar metricas en dict
    metrics_fp_ycbv = {
        'add_mean_mm': float(add_errors.mean() * 1000),
        'add_median_mm': float(np.median(add_errors) * 1000),
        'adds_mean_mm': float(adds_errors.mean() * 1000),
        'adds_median_mm': float(np.median(adds_errors) * 1000),
        'auc_add': float(auc_add),
        'auc_adds': float(auc_adds),
        'n_evaluated': len(add_errors),
    }
    for th in thresholds_mm:
        metrics_fp_ycbv[f'recall_add_{th}mm'] = float((add_errors * 1000 < th).mean())
        metrics_fp_ycbv[f'recall_adds_{th}mm'] = float((adds_errors * 1000 < th).mean())
else:
    print("Sin predicciones para calcular metricas.")
    metrics_fp_ycbv = {}

## 7. Visualizar ejemplos

In [ ]:
import matplotlib.pyplot as plt
from src.utils.visualization import draw_pose_axes

if results_ycbv:
    n_show = min(8, len(results_ycbv))
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()
    
    for idx in range(n_show):
        r = results_ycbv[idx]
        sample = ycbv.load_sample(r['scene_id'], r['img_id'])
        
        rgb_vis = sample['rgb'].copy()
        R = np.array(r['R_pred'])
        t = np.array(r['t_pred'])
        K = sample['cam_K']
        
        # Dibujar ejes de pose sobre la imagen
        try:
            rgb_vis = draw_pose_axes(rgb_vis, R, t, K, scale=0.05)
        except Exception:
            pass
        
        axes[idx].imshow(rgb_vis)
        axes[idx].set_title(
            f"S{r['scene_id']}/I{r['img_id']}\n"
            f"Obj {r['obj_id']} | {r['time_s']*1000:.0f}ms",
            fontsize=10
        )
        axes[idx].axis('off')
    
    # Ocultar ejes vacios
    for idx in range(n_show, len(axes)):
        axes[idx].axis('off')
    
    plt.suptitle('FoundationPose — YCB-Video Predictions', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Sin resultados para visualizar.")

## 8. Ejecutar en T-LESS

In [ ]:
tless_test_path = Path(f"{DATA_DIR}/tless/test_primesense")
TLESS_CKPT_FILE = f"{CHECKPOINT_DIR}/fp_tless_checkpoint.json"

# Retomar desde checkpoint si existe
results_tless = []
completed_tless_scenes = set()
timing_tless = 0
n_tless_obj = 0

if os.path.exists(TLESS_CKPT_FILE):
    with open(TLESS_CKPT_FILE) as f:
        ckpt = json.load(f)
    results_tless = ckpt["results"]
    completed_tless_scenes = set(ckpt["completed_scenes"])
    timing_tless = ckpt["timing_total"]
    n_tless_obj = ckpt["n_objects_evaluated"]
    print(f"CHECKPOINT T-LESS: {len(completed_tless_scenes)} escenas ya completadas\n")

if tless_test_path.exists() and tless_meshes:
    tless = BOPDataset(f"{DATA_DIR}/tless", split="test_primesense")
    tless_scenes = tless.get_scene_ids()
    print(f"T-LESS: {len(tless_scenes)} escenas de test\n")

    MAX_TLESS_SCENES = 3
    MAX_TLESS_IMAGES = 30
    eval_tless = tless_scenes[:MAX_TLESS_SCENES]
    tless_estimator_cache = {}

    for scene_idx, scene_id in enumerate(tqdm(eval_tless, desc="Escenas T-LESS")):
        if scene_id in completed_tless_scenes:
            continue

        n_images = tless.get_num_images(scene_id)
        scene_gt = tless.load_scene_gt(scene_id)

        image_ids = tless.get_image_ids(scene_id)
        for img_id in image_ids[:MAX_TLESS_IMAGES]:
            try:
                sample = tless.load_sample(scene_id, img_id)
                rgb = sample["rgb"]
                depth = sample["depth"]
                cam_K = sample["cam_K"]

                img_key = str(img_id)
                gt_list = scene_gt.get(img_key, scene_gt.get(img_id, []))

                for gt_idx, gt in enumerate(gt_list):
                    obj_id = gt["obj_id"]
                    if obj_id not in tless_meshes:
                        continue

                    mask_path = Path(DATA_DIR) / "tless" / "test_primesense" / scene_id / "mask_visib" / f"{img_id:06d}_{gt_idx:06d}.png"
                    if mask_path.exists():
                        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
                        mask = (mask > 0).astype(np.uint8)
                    else:
                        continue

                    if obj_id not in tless_estimator_cache:
                        tless_estimator_cache[obj_id] = create_estimator(
                            tless_meshes[obj_id], scorer, refiner, glctx
                        )
                    est = tless_estimator_cache[obj_id]

                    t0 = time.time()
                    pose_4x4 = est.register(
                        K=cam_K, rgb=rgb, depth=depth,
                        ob_mask=mask, iteration=REGISTER_ITERATIONS,
                    )
                    elapsed = time.time() - t0
                    timing_tless += elapsed
                    n_tless_obj += 1

                    results_tless.append({
                        "scene_id": int(scene_id),
                        "img_id": int(img_id),
                        "obj_id": int(obj_id),
                        "R_pred": pose_4x4[:3, :3].tolist(),
                        "t_pred": pose_4x4[:3, 3].tolist(),
                        "time_s": elapsed,
                    })
            except Exception as e:
                continue

        completed_tless_scenes.add(scene_id)

        # CHECKPOINT
        ckpt_data = {
            "results": results_tless,
            "completed_scenes": list(completed_tless_scenes),
            "timing_total": timing_tless,
            "n_objects_evaluated": n_tless_obj,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        }
        with open(TLESS_CKPT_FILE, "w") as f:
            json.dump(ckpt_data, f)
        print(f"  [CKPT] {len(completed_tless_scenes)} escenas, {n_tless_obj} objetos")

    avg_tless = timing_tless / max(n_tless_obj, 1)
    print(f"\n{'=' * 60}")
    print(f"  FoundationPose T-LESS — Resumen")
    print(f"{'=' * 60}")
    print(f"  Objetos evaluados:     {n_tless_obj}")
    print(f"  Predicciones totales:  {len(results_tless)}")
    print(f"  Tiempo promedio/obj:   {avg_tless*1000:.1f} ms")
    print(f"  Checkpoint:            {TLESS_CKPT_FILE}")
    print(f"{'=' * 60}")
else:
    print("T-LESS no disponible. Ejecuta 00_colab_setup.ipynb para descargar.")


## 9. Guardar resultados

In [ ]:
from datetime import datetime

base_dir = "/content/drive/MyDrive/TFM/experiments" if USE_GDRIVE else f"{REPO_DIR}/experiments"
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# --- Guardar YCB-V ---
if results_ycbv:
    exp_dir = f"{base_dir}/foundationpose_ycbv"
    os.makedirs(exp_dir, exist_ok=True)
    
    output_ycbv = {
        'method': 'FoundationPose',
        'paper': 'Wen et al., CVPR 2024',
        'dataset': 'ycbv',
        'timestamp': timestamp,
        'gpu': torch.cuda.get_device_name(0),
        'config': {
            'register_iterations': REGISTER_ITERATIONS,
            'max_scenes': MAX_SCENES,
            'max_images_per_scene': MAX_IMAGES_PER_SCENE,
        },
        'summary': {
            'n_predictions': len(results_ycbv),
            'n_images': n_images_evaluated,
            'n_objects': n_objects_evaluated,
            'avg_time_ms': avg_time * 1000,
        },
        'metrics': metrics_fp_ycbv,
        'predictions': results_ycbv,
    }
    
    result_file = f"{exp_dir}/results_{timestamp}.json"
    with open(result_file, 'w') as f:
        json.dump(output_ycbv, f, indent=2)
    print(f"[OK] YCB-V guardado: {result_file}")

# --- Guardar T-LESS ---
if results_tless:
    exp_dir_tless = f"{base_dir}/foundationpose_tless"
    os.makedirs(exp_dir_tless, exist_ok=True)
    
    output_tless = {
        'method': 'FoundationPose',
        'dataset': 'tless',
        'timestamp': timestamp,
        'gpu': torch.cuda.get_device_name(0),
        'n_predictions': len(results_tless),
        'predictions': results_tless,
    }
    
    result_file_tless = f"{exp_dir_tless}/results_{timestamp}.json"
    with open(result_file_tless, 'w') as f:
        json.dump(output_tless, f, indent=2)
    print(f"[OK] T-LESS guardado: {result_file_tless}")

storage = 'Google Drive' if USE_GDRIVE else 'Colab Storage'
print(f"\nResultados persistidos en {storage}.")
print("Continua con 02_gdrnet_eval.ipynb para el baseline comparativo.")